# Linear Regression & GLMs — Student Lab

Complete all TODOs. Avoid sklearn for core parts.

In [1]:
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Section 0 — Synthetic Dataset (with collinearity)
We generate data where features can be highly correlated to motivate ridge.

In [2]:
def make_regression(n=400, d=5, noise=0.5, collinear=True):
    X = rng.standard_normal((n, d))
    if collinear and d >= 2:
        X[:, 1] = X[:, 0] * 0.95 + 0.05 * rng.standard_normal(n)
    w_true = rng.standard_normal(d)
    y = X @ w_true + noise * rng.standard_normal(n)
    return X, y, w_true

X, y, w_true = make_regression()
n, d = X.shape
check('shapes', y.shape == (n,))
print('corr(x0,x1)=', np.corrcoef(X[:,0], X[:,1])[0,1])

OK: shapes
corr(x0,x1)= 0.9985704465455701


## Section 1 — OLS Closed Form

### Task 1.1: Closed-form w_hat using solve

# TODO: compute w_hat using solve on (X^T X) w = X^T y
# HINT: `XtX = X.T@X`, `Xty = X.T@y`, `np.linalg.solve(XtX, Xty)`

**Checkpoint:** Why is explicit inverse discouraged?

**Answer:** Explicit inverse is discouraged because it’s slower and less numerically stable. Solving the linear system directly avoids extra error, especially when features are highly correlated and X^\top X is ill-conditioned.

In [3]:
# TODO
XtX = X.T @ X
Xty = X.T @ y
w_hat = np.linalg.solve(XtX, Xty)

check('w_shape', w_hat.shape == (d,))

OK: w_shape


### Task 1.2: Evaluate fit + residuals
Compute:
- predictions y_pred
- MSE
- residual mean and std

**Interview Angle:** What does a structured residual pattern imply (e.g., nonlinearity)?

**Answer:** A structured pattern in the residuals means the model is missing something systematic in the data. Instead of looking like random noise, the errors follow a shape or trend, which tells you the model is not capturing the true relationship.

In [4]:
# TODO
y_pred = X @ w_hat
mse = float(np.mean((y_pred - y)**2))
resid = y_pred - y
print('mse', mse, 'resid_mean', resid.mean(), 'resid_std', resid.std())
check('finite', np.isfinite(mse))

mse 0.2289243168902079 resid_mean 0.02930128510763731 resid_std 0.47756230125633753
OK: finite


## Section 2 — Gradient Descent

### Task 2.1: Implement MSE loss + gradient

Loss = mean((Xw-y)^2), grad = (2/n) X^T(Xw-y)

# TODO: implement `mse_loss_and_grad`

**FAANG gotcha:** shapes and constants.

In [5]:
def mse_loss_and_grad(X, y, w):
    # TODO
    loss = float(np.mean((X @ w - y)**2))
    grad = (2.0 / X.shape[0]) * X.T @ (X @ w - y)
    return loss, grad

w0 = np.zeros(d)
loss0, g0 = mse_loss_and_grad(X, y, w0)
check('grad_shape', g0.shape == (d,))
check('finite_loss', np.isfinite(loss0))

OK: grad_shape
OK: finite_loss


### Task 2.2: Train with GD + compare to closed-form

# TODO: implement a simple GD loop, track loss, and compare final weights to w_hat.

**Checkpoint:** How does feature scaling affect GD?

**Answer:** Feature scaling has a huge effect on how fast and stable GD converges. Without feature scaling GD keeps overshooting in steep directions and barely moving in flat ones.
But with feature scaling makes gradient descent converge faster and more stably by preventing some features from dominating the gradient and creating ill-conditioned loss surfaces.

In [6]:
def train_gd(X, y, lr=0.05, steps=500):
    # TODO
    w = np.zeros(X.shape[1])
    losses = []
    for _ in range(steps):
      loss, g = mse_loss_and_grad(X, y, w)
      losses.append(loss)
      w = w - lr * g
    return w, losses

w_gd, losses = train_gd(X, y, lr=0.05, steps=500)
print('final_loss', losses[-1])
print('||w_gd-w_hat||', np.linalg.norm(w_gd - w_hat))
check('loss_decreases', losses[-1] <= losses[0])

final_loss 0.23136626716013967
||w_gd-w_hat|| 1.376975427170427
OK: loss_decreases


## Section 3 — Ridge Regression (L2)

### Task 3.1: Ridge closed-form
w = (X^T X + λI)^{-1} X^T y

# TODO: implement ridge_solve

**Interview Angle:** Why does ridge help under collinearity?

**Answer:** Ridge helps under collinearity by stabilizing the matrix inversion and shrinking weights, preventing large, unstable coefficients when features are highly correlated.

When features are highly correlated the model can’t decide how much credit to give to each similar feature, so it overreacts.
- Columns of X carry almost the same information
- X.T @ X becomes almost singular (hard to invert)
- Small noise in data causes huge swings in weights

Ridge fixes this by penalizing large weights. Ridge adds a “cost” for large weights, so the model stops over-relying on any one of several similar features.

What adding lambda*I does:
- Stabilizes the math: It makes X.T @ X + lambda * I safely invertible, even when features are almost duplicates.
- Shrinks weights: The model prefers spreading weight more evenly instead of making any one weight huge.
- Reduces sensitivity to noise: Small changes in data no longer cause big changes in weights.

In [8]:
def ridge_solve(X, y, lam):
    # TODO
    d = X.shape[1]
    return np.linalg.solve(X.T @ X + lam * np.eye(d), X.T @ y)

w_ridge = ridge_solve(X, y, lam=1.0)
check('ridge_shape', w_ridge.shape == (d,))

OK: ridge_shape


### Task 3.2: Bias/variance demo with train/test split

# TODO: split into train/test and compare MSE for multiple lambdas.

**Checkpoint:** why can test error improve even when train error worsens?

**Answer:** Test error can improve even when training error gets worse because the model is generalizing better instead of memorizing the training data.
Basically,
- Increasing lambda (ridge regularization) restricts model complexity.
- The model fits the training data less perfectly, so training error goes up.
- But it becomes less sensitive to noise and collinearity, so it performs better on unseen data.


In [9]:
# TODO
idx = rng.permutation(n)
train = idx[: int(0.7*n)]
test = idx[int(0.7*n):]
Xtr, ytr = X[train], y[train]
Xte, yte = X[test], y[test]

lams = [0.0, 0.1, 1.0, 10.0]
results = []
for lam in lams:
    w = ridge_solve(Xtr, ytr, lam=lam) if lam > 0 else np.linalg.solve(Xtr.T@Xtr, Xtr.T@ytr)
    tr_mse = np.mean((Xtr@w - ytr)**2)
    te_mse = np.mean((Xte@w - yte)**2)
    results.append((lam, tr_mse, te_mse))
print('lam, train_mse, test_mse')
for r in results:
    print(r)

lam, train_mse, test_mse
(0.0, np.float64(0.21799841512243526), np.float64(0.25774539368101873))
(0.1, np.float64(0.21809342046539706), np.float64(0.25841616068584095))
(1.0, np.float64(0.21911091721949197), np.float64(0.26106348492134784))
(10.0, np.float64(0.22348087752854687), np.float64(0.2703372913154806))


## Section 4 — GLM Intuition

### Task 4.1: Match tasks to (distribution, link)
Fill in a table for:
- regression
- binary classification
- count prediction

**Explain:** what changes when you go from OLS to a GLM?

**Answer:** OLS is a GLM with a Gaussian distribution and identity link; moving to a GLM means changing the assumed output distribution, the link function, and therefore the loss, while keeping a linear predictor.
- Output distribution
	- OLS assumes Gaussian errors
	- GLMs allow Bernoulli, Poisson
- Link function
	- OLS: identity link
	- GLM: sigmoid, log, to enforce valid outputs
- Loss function
	- OLS -> MSE
	- GLM -> negative log-likelihood of chosen distribution

| Problem | Target type | Distribution | Link | Loss |
|---|---|---|---|---|
| House price | continuous | Gaussian (Normal) | Identity | MSE |
| Fraud | binary | Bernoulli | Logit (Sigmoid) | Binary Cross Entropy |
| Clicks per user | count | Poisson | Exp | Poisson negative log |


---
## Submission Checklist
- All TODOs completed
- Train/test results shown for ridge
- Short answers to checkpoint questions
